# End-to-End Demo (thin orchestration over `src/`)

This notebook contains **no experimental logic** — it only calls the tested,
importable functions in `src/`. All real work lives in the package and is
configured via `configs/`. For full runs prefer the CLIs / `make reproduce`.

> Numbers produced here are real only if pointed at the real Kermany dataset.
> The synthetic smoke-test data is for pipeline verification only.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # make `src` importable

from src.config import load_config
from src.dataset import build_dataloaders, validate_dataset, compute_dataset_statistics
from src.models import build_model

# Point data.root at your dataset; here we keep defaults from the config.
cfg = load_config('../configs/efficientnet_b0.yaml', overrides=['device=cpu'])
cfg.experiment_name, cfg.model.name

In [ ]:
# Dataset statistics + integrity (requires data present at cfg.data.root)
compute_dataset_statistics(cfg)

In [ ]:
# Build the model and inspect parameter count
from src.utils import count_parameters
model = build_model(name=cfg.model.name, pretrained=cfg.model.pretrained,
                    freeze_stages=cfg.model.freeze_stages)
total, trainable = count_parameters(model)
print(f'Total params: {total:,} | Trainable: {trainable:,}')

## Train / evaluate / quantize / explain

These are long-running; prefer the CLIs. The calls below mirror `make reproduce`.

In [ ]:
from src.train import train
from src.evaluate import evaluate_checkpoint
from src.quantize import run_quantization_study
from src.explainability import run_explainability
from src.reporting import build_all_reports

result = train(cfg)
ckpt = result['checkpoint']
evaluate_checkpoint(ckpt, cfg=cfg, device_str='cpu')
run_quantization_study(ckpt, cfg)
run_explainability(ckpt, cfg=cfg, device_str='cpu')
build_all_reports(cfg)